# Урок 33. Итоговый проект и защита

11 класс · IV четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 32](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-32.ipynb) · [Урок 34 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-34.ipynb)

---

Финальный проект: содержательная задача, реализация, результат. Представление работ и рецензирование.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 11А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
for адрес in ("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py",
              "https://cdn.jsdelivr.net/gh/LeksssGray/informatika-7-11@main/lib/schoolinf.py"):
    try:
        urllib.request.urlretrieve(адрес, "schoolinf.py")
        break
    except Exception as ошибка:
        print("⚠️ Не скачалась библиотека проверки:", ошибка)
else:
    raise SystemExit("Проверь интернет и запусти эту ячейку ещё раз. "
                     "Не помогло — покажи это сообщение учителю.")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-33", name=ФИО, klass=Класс)

## Что делаем

Итоговый проект школы — содержательная задача, решённая честно:
с постановкой, проверкой и выводом. Разберём, как устроен хороший
проект, пройдём все шаги на примере и договоримся о критериях.

## Проект

### Четыре части

1. **Постановка.** Что решаем, для кого и почему это не тривиально.
   Вход и выход описаны точно: «на входе CSV с оценками, на выходе
   средние по классам и самый сильный предмет».
2. **Реализация.** Код или таблица, которые делают работу. Функции
   с понятными именами, данные отдельно от логики.
3. **Проверка.** Как ты убедился, что ответ верный: набор случаев
   с известными ответами, сравнение с медленным, но заведомо верным
   решением, проверка на крайних данных.
4. **Вывод.** Что получилось, где решение ограничено, что улучшить.

### Направления

* **анализ данных:** CSV, группировка, средние и доли, график, вывод
  с оговорками;
* **алгоритм:** решатель задач ЕГЭ одного типа (маски, игры, файлы)
  с проверкой на всех примерах и оценкой сложности;
* **веб:** страница или небольшой сайт на HTML и CSS с макетом
  из Figma;
* **модель:** вклад, эпидемия, очередь — с параметрами
  и экспериментом «что если».

### Критерии

| Критерий | 0 | 1 | 2 |
|---|---|---|---|
| постановка | нет | расплывчатая | точный вход и выход |
| реализация | не работает | работает с ошибками | работает, код понятный |
| проверка | нет | несколько случаев | набор с крайними случаями или сверка с перебором |
| вывод | нет | не следует из результатов | опирается на результаты, названы ограничения |
| защита | не рассказал | без демонстрации | показал, ответил про сложность |

### Защита — пять минут

Постановка → как решал → как проверял → что получилось → что дальше.
Почти наверняка спросят: «Сколько это будет работать на миллионе
записей?» — подготовь ответ заранее.

## Смотрим, как это работает

### Постановка

Дан ряд из n чисел — например, продажи по дням — и q вопросов вида
«сколько продали с дня l по день r?». Нужно отвечать быстро, даже если
и дней, и вопросов сотни тысяч.

### Решение в лоб — и сколько оно стоит

Каждый вопрос — сумма отрезка, до n шагов. Всего до n · q шагов.

In [ ]:
import random
import time

random.seed(33)
n, q = 20000, 2000
продажи = [random.randint(0, 100) for _ in range(n)]
вопросы = [sorted((random.randrange(n), random.randrange(n))) for _ in range(q)]

старт = time.perf_counter()
ответы_в_лоб = [sum(продажи[l:r + 1]) for l, r in вопросы]
print(f"В лоб: {time.perf_counter() - старт:.3f} с")

### Идея: префиксные суммы

Посчитаем один раз `p[i]` — сумму первых i чисел. Тогда сумма отрезка
`[l, r]` — это `p[r + 1] − p[l]`: одно вычитание на вопрос.
Всего n + q шагов вместо n · q.

In [ ]:
старт = time.perf_counter()
p = [0]
for x in продажи:
    p.append(p[-1] + x)
ответы_быстро = [p[r + 1] - p[l] for l, r in вопросы]
print(f"Префиксы: {time.perf_counter() - старт:.3f} с")

### Проверка

Быстрое решение сверяем с медленным, в правильности которого
не сомневаемся. Это главный приём проверки алгоритмов.

In [ ]:
print("Ответы совпали:", ответы_в_лоб == ответы_быстро)

for l, r in [(0, 0), (0, n - 1), (n - 1, n - 1)]:   # крайние случаи
    assert p[r + 1] - p[l] == sum(продажи[l:r + 1])
print("Крайние случаи в порядке")

### Вывод

«Префиксные суммы отвечают на вопрос за одно действие после
подготовки за n шагов. На 20 000 дней и 2 000 вопросов это заметно
быстрее, и разница растёт с данными: в лоб — O(n · q), с префиксами —
O(n + q). Ограничение: если продажи меняются, массив префиксов
придётся пересчитывать».

## Пробуем сами

### Задача 1. Префиксные суммы

Функция получает список чисел и возвращает список, в котором на месте i стоит сумма первых i + 1 элементов.

In [ ]:
def префиксные_суммы(числа):
    return ...

In [ ]:
si.check("1", префиксные_суммы, [
    ([1, 2, 3], [1, 3, 6]),
    ([], []),
    ([5], [5]),
    ([-1, 1], [-1, 0]),
])

### Задача 2. Что важнее в защите

Что важнее всего в защите проекта?

In [ ]:
#@title 🧩 Задача 2. Что важнее в защите { display-mode: "form" }
#@markdown Выбери ответ
ответ = "выбери ответ" #@param ["выбери ответ", "количество слайдов", "вывод, основанный на данных", "громкость голоса"]

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("2", ответ, "5bdafd5f1812b8c3",
         hint="Выводы должны опираться на то, что ты проверил.")

### Задача 3. Лучшее окно

Функция получает список продаж и число k и возвращает наибольшую сумму
k подряд идущих дней. Используй префиксные суммы, а не `sum` в цикле.

In [ ]:
def лучшее_окно(числа, k):
    return ...

In [ ]:
si.check("3", лучшее_окно, [
    (([1, 3, 2, 5, 1], 2), 7),
    (([5], 1), 5),
    (([1, 2, 3], 3), 6),
    (([4, -1, -1, 4], 2), 3),
])

### Задача 4. Сверка с перебором

Напиши медленную, но очевидную функцию `лучшее_окно_перебором` —
через `sum` каждого окна. В проекте такая функция служит эталоном
для проверки быстрой.

In [ ]:
def лучшее_окно_перебором(числа, k):
    return ...

In [ ]:
si.check("4", лучшее_окно_перебором, [
    (([1, 3, 2, 5, 1], 2), 7),
    (([4, -1, -1, 4], 2), 3),
    (([2, 2, 2], 1), 2),
])

### Вопрос 5. Миллион на миллион

Дней миллион и вопросов миллион. Сколько примерно шагов сделает
решение в лоб?

In [ ]:
#@title 🧩 Вопрос 5. Оценка сложности { display-mode: "form" }
#@markdown Выбери ответ
ответ = "выбери ответ" #@param ["выбери ответ", "около двух миллионов", "около миллиона миллионов", "около тысячи"]

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("5", ответ, "36c43155ea2f1ff6",
         hint="n · q, где оба — миллион.")

---

## Домашнее задание

### 1. Проект

Реализуй проект по четырём частям и оформи отчёт в ноутбуке —
по шаблону ниже. Подготовь выступление на пять минут и ответы
на два вопроса о сложности своего решения.

**Постановка:** что решаю, вход, выход …

**Реализация:** главные функции и что они делают …

**Проверка:** набор случаев, сверка с перебором …

**Вывод и ограничения:** …

**Сложность:** O(…), потому что …

In [ ]:
# реализация и проверка

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 32](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-32.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 34 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-34.ipynb)